### Vertox: Fast Option Pricing using Fourier Transform


SDE (stochastic differential equation)

Under risk-neutral measure, the asset price follows GBM (geometric brownian motion)

$$dS_t = r S_t dt + \sigma S_t dW_t$$

$r$ is the risk-free interest rate.

$\sigma$ is the asset volatility.

$dW_t$ is a standard Brownian motion increment ($dW_t \sim \mathcal{N}(0, dt)$).


By applying Itô's Lemma to $X_t = \log S_t$, we get the exact analytical solution to simulate the terminal asset price $S_T$ at maturity $T$ in a single time step:

$$S_T = S_0 \exp\left( \left(r - \frac{1}{2}\sigma^2\right)T + \sigma \sqrt{T} Z \right)$$

Where $Z \sim \mathcal{N}(0, 1)$ is a standard normal random variable. The expected payoff of a European Call option in frequency space is discounted back to today:


The fair value of the European option is the expected value ($\mathbb{E}$) of its terminal payoff under the risk-neutral measure ($\mathbb{Q}$), discounted back to present value.


$$\text{Price} = e^{-rT} \mathbb{E}^\mathbb{Q}[\max(S_T - K, 0)]$$


The statistical precision of a Monte Carlo simulation scales inversely with the square root of the number of simulated paths ($N$).

$$\mathcal{O}\left(\frac{1}{\sqrt{N}}\right)$$


In [ ]:
import numpy as np
from typing import Final


def monte_carlo_call_price(
    s0: float, k: float, t: float, r: float, sigma: float, num_paths: int = 1_000_000
) -> dict[str, float]:
    """Prices a European Call Option using a vectorized Monte Carlo simulation.

    Conforms to Python 3.11+ native generic typing standards.
    """
    # Seed the generator for reproducibility
    rng = np.random.default_rng(seed=42)

    # 1. Simulate the terminal state space of the asset using the analytic solution of GBM
    # Time complexity: O(N), Space complexity: O(N) allocations for the array
    z = rng.standard_normal(size=num_paths)

    drift = (r - 0.5 * sigma**2) * t
    diffusion = sigma * np.sqrt(t) * z
    s_t = s0 * np.exp(drift + diffusion)

    # 2. Calculate the payoff vector: max(S_T - K, 0)
    payoffs = np.maximum(s_t - k, 0.0)

    # 3. Discount the expectation back to present value
    discount_factor: Final[float] = np.exp(-r * t)
    estimated_price = discount_factor * np.mean(payoffs)

    # 4. Statistical error bounds (Standard Error of the Mean Estimate)
    # Variance profile scales inversely with the square root of N (O(1/sqrt(N)))
    std_error = discount_factor * np.std(payoffs, ddof=1) / np.sqrt(num_paths)

    return {"price": float(estimated_price), "std_error": float(std_error)}


# Run the simulation engine
if __name__ == "__main__":
    # Test Parameters
    S0 = 100.0  # Spot Price
    K = 100.0  # Strike Price
    T = 1.0  # Time to maturity (1 year)
    R = 0.05  # Risk-free rate (5%)
    SIGMA = 0.20  # Volatility (20%)
    PATHS = 5_000_000

    result = monte_carlo_call_price(S0, K, T, R, SIGMA, num_paths=PATHS)

    print(f"Monte Carlo Call Price: ${result['price']:.4f}")
    print(f"Standard Error Bounds:  +/- ${result['std_error']:.4f}")

Monte Carlo Call Price: $10.4588
Standard Error Bounds:  +/- $0.0066


In [ ]:
### Here is a simple implementation of the Heston model in Python using the Euler-Maruyama method for numerical simulation:

import numpy as np


def heston_model(
    s0: float,
    v0: float,
    kappa: float,
    theta: float,
    sigma: float,
    rho: float,
    t: float,
    r: float,
    num_paths: int = 1_000_000,
) -> dict[str, float]:
    """Simulates the Heston model for option pricing.

    Conforms to Python 3.11+ native generic typing standards.
    """
    # Seed the generator for reproducibility
    rng = np.random.default_rng(seed=42)

    # Time step
    dt = t / 252  # Assuming 252 trading days in a year

    # Initialize arrays for asset price and variance
    s_t = np.zeros(num_paths)
    v_t = np.zeros(num_paths)

    s_t[0] = s0
    v_t[0] = v0

    # Simulate paths using Euler-Maruyama method
    for i in range(1, num_paths):
        z1 = rng.standard_normal()
        z2 = rng.standard_normal()

        # Correlated Brownian motions
        dw_s = z1 * np.sqrt(dt)
        dw_v = (rho * z1 + np.sqrt(1 - rho**2) * z2) * np.sqrt(dt)

        # Update variance using the Heston model dynamics
        v_t[i] = np.maximum(
            v_t[i - 1]
            + kappa * (theta - v_t[i - 1]) * dt
            + sigma * np.sqrt(v_t[i - 1]) * dw_v,
            0,
        )

        # Update asset price using the Heston model dynamics
        s_t[i] = s_t[i - 1] * np.exp(
            (r - 0.5 * v_t[i - 1]) * dt + np.sqrt(v_t[i - 1]) * dw_s
        )

    # Calculate the payoff vector: max(S_T - K, 0)
    payoffs = np.maximum(s_t - k, 0.0)

    # Discount the expectation back to present value
    discount_factor: Final[float] = np.exp(-r * t)
    estimated_price = discount_factor * np.mean(payoffs)

    # Statistical error bounds (Standard Error of the Mean Estimate)
    std_error = discount_factor * np.std(payoffs, ddof=1) / np.sqrt(num_paths)

    return {"price": float(estimated_price), "std_error": float(std_error)}
    print(f"Heston Model Call Price: ${result['price']:.4f}")
    print(f"Standard Error Bounds:  +/- ${result['std_error']:.4f}")


print(f"Heston Model Call Price: ${result['price']:.4f}")
print(f"Standard Error Bounds:  +/- ${result['std_error']:.4f}")

Heston Model Call Price: $10.4588
Standard Error Bounds:  +/- $0.0066


In [ ]:
import numpy as np
from typing import Final
from scipy.integrate import trapezoid

def gbm_characteristic_function(
    z: np.ndarray, s0: float, t: float, r: float, sigma: float
) -> np.ndarray:
    """Standard analytical Characteristic Function for log(S_T)."""
    # The true risk-neutral drift for log-price
    mu: Final[float] = np.log(s0) + (r - 0.5 * sigma**2) * t
    variance: Final[float] = (sigma**2) * t
    return np.exp(1j * z * mu - 0.5 * variance * (z**2))

def lewis_fourier_call_price(
    s0: float,
    k: float,
    t: float,
    r: float,
    sigma: float,
    u_max: float = 1000.0,
    num_points: int = 100000,
) -> float:
    # 1. Frequency grid spacing
    u = np.linspace(1e-5, u_max, num_points)

    # 2. CORRECTED: Symmetrical upper complex plane shift (z = u + 0.5j)
    z = u + 0.5j

    # 3. Evaluate the Characteristic Function at the shifted contour
    phi = gbm_characteristic_function(z, s0, t, r, sigma)

    # 4. Evaluate the correct Lewis option pricing integrand
    log_k: Final[float] = np.log(k)
    integrand = np.real(np.exp(-1j * u * log_k) * phi / (u**2 + 0.25))

    # 5. Integrate using the trapezoidal rule
    integral_value = trapezoid(integrand, u)

    # 6. Apply the final pricing framework
    multiplier: Final[float] = np.exp(-r * t) / np.pi

    # Corrected reconstruction architecture
    call_price = s0 - (multiplier * np.sqrt(s0 * k) * integral_value)
    return float(call_price)

if __name__ == "__main__":
    S0 = 100.0
    K = 100.0
    T = 1.0
    R = 0.05
    SIGMA = 0.20

    fourier_price = lewis_fourier_call_price(
        s0=S0, k=K, t=T, r=R, sigma=SIGMA,
        u_max=1000.0,
        num_points=100000
    )

    print(f"--- FOURIER INTEGRATION ENGINE (CONVERGED) ---")
    print(f"Fourier Calculated Price: ${fourier_price:.6f}")

--- FOURIER INTEGRATION ENGINE (CONVERGED) ---
Fourier Calculated Price: $91.290028
